In [ ]:
import warnings
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error

from sktime.forecasting.base import ForecastingHorizon
from sktime.forecasting.arima import AutoARIMA
from sktime.forecasting.ets import AutoETS

from prophet import Prophet

warnings.filterwarnings("ignore")

In [ ]:
def evaluate_dataset(csv_path: str):
  df = pd.read_csv(csv_path)
  df["Date"] = pd.to_datetime(df["Date"])
  df = df.sort_values("Date")
  y = df.set_index("Date")["Revenue"].astype(float)

  cutoff = "2011-08-31"
  y_train = y.loc[:cutoff]
  y_test  = y.loc["2011-09-01":]

  fh = ForecastingHorizon(np.arange(1, len(y_test) + 1), is_relative=True)

  print("=" * 80)
  print(csv_path)
  print(f"TRAIN: {len(y_train)} | TEST: {len(y_test)}")
  print("=" * 80)

  print("\nFitting AutoARIMA...")
  arima = AutoARIMA(seasonal=True, sp=7, d=None, information_criterion="aic", suppress_warnings=True, error_action="ignore")
  arima.fit(y_train)
  arima_pred = arima.predict(fh)
  arima_mae  = mean_absolute_error(y_test, arima_pred)
  arima_rmse = np.sqrt(mean_squared_error(y_test, arima_pred))

  print("\nFitting AutoETS...")
  ets = AutoETS(auto=True, sp=7)
  ets.fit(y_train)
  ets_pred  = ets.predict(fh)
  ets_mae   = mean_absolute_error(y_test, ets_pred)
  ets_rmse  = np.sqrt(mean_squared_error(y_test, ets_pred))

  print("\nFitting Prophet...")
  has_two_years = len(y_train) >= 365 * 2
  df_prophet_train = pd.DataFrame({"ds": y_train.index, "y": y_train.values})

  prophet_param_grid = [
    {"changepoint_prior_scale": cp, "seasonality_prior_scale": sp}
    for cp in [0.01, 0.1, 0.5]
    for sp in [1.0, 10.0]
  ]

  fh_horizon = max(int(len(y_test) * 0.25), 30)
  n_splits   = 3
  window     = int(len(df_prophet_train) * 0.60)
  step       = max((len(df_prophet_train) - window) // n_splits, 1)

  best_prophet_params = None
  best_prophet_score  = np.inf

  for params in prophet_param_grid:
    fold_maes = []
    for k in range(n_splits):
      split_idx  = window + k * step
      train_fold = df_prophet_train.iloc[:split_idx]
      val_fold   = df_prophet_train.iloc[split_idx:split_idx + fh_horizon]
      if len(val_fold) == 0:
        continue
      m      = Prophet(weekly_seasonality=True, daily_seasonality=False, yearly_seasonality=has_two_years, **params)
      m.fit(train_fold)
      future = m.make_future_dataframe(periods=len(val_fold))
      fc     = m.predict(future).tail(len(val_fold))
      fold_maes.append(mean_absolute_error(val_fold["y"].values, fc["yhat"].values))
    avg_mae = np.mean(fold_maes)
    if avg_mae < best_prophet_score:
      best_prophet_score  = avg_mae
      best_prophet_params = params

  print("Best Prophet params:", best_prophet_params)
  prophet_model = Prophet(weekly_seasonality=True, daily_seasonality=False, yearly_seasonality=has_two_years, **best_prophet_params)
  prophet_model.fit(df_prophet_train)
  prophet_fc   = prophet_model.predict(pd.DataFrame({"ds": y_test.index}))
  prophet_pred = prophet_fc["yhat"].values
  prophet_mae  = mean_absolute_error(y_test, prophet_pred)
  prophet_rmse = np.sqrt(mean_squared_error(y_test, prophet_pred))

  print(f"\nAutoARIMA  MAE={arima_mae:.4f}  RMSE={arima_rmse:.4f}")
  print(f"AutoETS    MAE={ets_mae:.4f}  RMSE={ets_rmse:.4f}")
  print(f"Prophet    MAE={prophet_mae:.4f}  RMSE={prophet_rmse:.4f}")

  return {
    "dataset":        csv_path,
    "arima_mae":      arima_mae,
    "arima_rmse":     arima_rmse,
    "ets_mae":        ets_mae,
    "ets_rmse":       ets_rmse,
    "prophet_mae":    prophet_mae,
    "prophet_rmse":   prophet_rmse,
    "prophet_params": best_prophet_params,
    "arima_pred":     arima_pred.tolist(),
    "ets_pred":       ets_pred.tolist(),
    "prophet_pred":   prophet_pred.tolist(),
    "n_pred":         len(y_test),
  }

In [ ]:
import os
os.makedirs("resultados", exist_ok=True)

DATASETS = ["product.csv", "country.csv", "customer.csv"]

serie_map = {"product": "Produto", "country": "País", "customer": "Cliente"}
rows = []

for dataset in DATASETS:
  res        = evaluate_dataset("data/" + dataset)
  serie_name = serie_map[dataset.replace(".csv", "")]

  for modelo, pred_key, mae_key, rmse_key in [
    ("AutoARIMA", "arima_pred",   "arima_mae",   "arima_rmse"),
    ("AutoETS",   "ets_pred",     "ets_mae",     "ets_rmse"),
    ("Prophet",   "prophet_pred", "prophet_mae", "prophet_rmse"),
  ]:
    rows.append({
      "serie":              serie_name,
      "periodos_previstos": res["n_pred"],
      "modelo":             modelo,
      "valores_previstos":  res[pred_key],
      "MAE":                round(res[mae_key],  2),
      "RMSE":               round(res[rmse_key], 2),
    })

df_results = pd.DataFrame(rows)
df_results.to_csv("resultados/resultados_estatisticos.csv", index=False)
print("Resultados salvos em resultados/resultados_estatisticos.csv")
print(df_results[["serie", "modelo", "MAE", "RMSE"]])